### Task 7: 

Create an interactive model of the virtual, enlarged image of an object placed inside the focal range of an ideal thin lens.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import numpy as np
import os
from PIL import Image

def load_img(image_path):
    """loads image file and converts it to a numpy array."""
    try:
        img = Image.open(image_path) 
        return np.array(img)
    except Exception as e:
        print(f"Error loading image: {e}")
        return None
    
def open_img():
    image_path = "stinkbug.png"  # replace this image file if needed
    if not os.path.exists(image_path): # error handling
        print(f"'{image_path}' not found")
        raise SystemExit
    else:
        img_path = load_img(image_path)
        if img_path is None:
            print('uhoh')
            raise SystemExit
        print(f"{image_path} successfully loaded")
        return img_path

orig_img = open_img()

def lens(X, Y, f): # lens equation
    xx = -(-1/X + 1/f)**(-1)
    yy = Y * xx / X
    return xx, yy

# scaling image up/down to fit graph
height, width = orig_img.shape[:2]
aspect_ratio = height / width
x0, y0 = 10 , 0    #Object centre
obj_w = 5       #Width of object (this scales the image)
f = 15          #focal length
# maximum and minimum x and y coords
x_min = x0 - obj_w/2
x_max = x0 + obj_w/2
y_max = obj_w * aspect_ratio / 2 + y0
y_min = -obj_w * aspect_ratio / 2 + y0
# meshgrid
x = np.linspace(0, obj_w, width) - obj_w/2 + x0
y = -np.linspace(0, obj_w*aspect_ratio, height) + obj_w*aspect_ratio/2 + y0
X, Y = np.meshgrid(x,y)  # Non-uniform spacing allowed
# figure and axis
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
thinlens = Ellipse(xy=(0, 0),  label = 'Thin Lens', width=3, height=20, edgecolor='b', fc = 'None', lw=0.5)
ax.add_patch(thinlens)
# original image
img_show = ax.pcolormesh(X, Y, orig_img, shading='auto', zorder=3)
#transformed image
xx,yy = lens(X, Y, f)
refl_show = ax.pcolormesh(xx, yy, orig_img, shading='auto', zorder=3)
# axis position, limits and labels
ax.spines['left'].set_position(('outward', 0.8)) # position of y-axis
ax.spines['bottom'].set_position(('outward', 0.8)) # position of x-axis
ax.set_xlim(-5, 55)
ax.set_ylim(-20, 20)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
ax.set_title('Thin Converging Lens: Inside Focal Range', fontsize=16)
# grid
ax.grid(True, alpha=0.5, linestyle='-', zorder = 1)
ax.set_aspect('equal', adjustable='box')
plt.legend(loc='upper left', fontsize=12)

def update(event): # update image position
    global X, Y, x_min, x_max, y_max, y_min
    # moving up
    if event.key == 'up':
        y_max += 0.5
        if y_max > 10: # HEIGHT OF LENS == 10
            print('out of bounds')
            y_max -= 0.5
        else:
            Y += 0.5
            y_min += 0.5
            move_pic()
    #moving down
    elif event.key == 'down':
        y_min -= 0.5
        if y_min < -10: # HEIGHT OF LENS == -10
            print('out of bounds')
            y_min += 0.5
        else:
            Y -= 0.5
            y_max -= 0.5
            move_pic()
    #moving left
    elif event.key == 'left':
        x_min -= 0.5
        if x_min <= 0:
            print('out of bounds')
            x_min += 0.5
        else:
            X -= 0.5
            x_max -= 0.5
            move_pic()
    #moving right
    elif event.key == 'right':
        x_max += 0.5
        if x_max >= f: # F == FOCAL LENGTH:
            print('out of bounds')
            x_max -= 0.5
        else:
            X += 0.5
            x_min += 0.5
            move_pic()
        X += 0.5
        move_pic()
 
def move_pic():
    # actually move the images
    global img_show, refl_show
    xx,yy = lens(X, Y, f) # re-calculating reflected img shape
    img_show.remove()
    refl_show.remove()
    img_show = ax.pcolormesh(X, Y, orig_img, shading='auto', zorder=3)
    refl_show = ax.pcolormesh(xx, yy, orig_img, shading='auto', zorder=3)
    plt.show()

fig.canvas.mpl_connect('key_press_event', update)
plt.show()